In [0]:
import random
from datetime import datetime

import pandas as pd

from config import SimulationConfig

In [0]:
class InventoryGenerator:

    def __init__(self):
        self.config = SimulationConfig()

    def generate_store_inventory(
        self,
        sales_df: pd.DataFrame
    ) -> pd.DataFrame:

        inventory_records = []

        grouped = (
            sales_df
            .groupby(
                [
                    "product_id",
                    "store_id"
                ]
            )
            ["sales_qty"]
            .mean()
            .reset_index()
        )

        for _, row in grouped.iterrows():

            days_of_cover = random.randint(
                self.config.MIN_DAYS_OF_COVER,
                self.config.MAX_DAYS_OF_COVER
            )

            inventory_qty = int(
                row["sales_qty"] * days_of_cover
            )

            inventory_records.append({
                "product_id":
                    row["product_id"],

                "store_id":
                    row["store_id"],

                "snapshot_date":
                    datetime.now().date(),

                "inventory_qty":
                    inventory_qty,

                "days_of_cover":
                    days_of_cover,

                "created_at":
                    datetime.now()
            })

        return pd.DataFrame(
            inventory_records
        )

    def generate_dc_inventory(
        self,
        store_inventory_df: pd.DataFrame,
        store_to_dc_map: dict
    ) -> pd.DataFrame:
        """
        Generate DC inventory based on stores served by each DC.
        
        Logic:
        - Each DC holds inventory for stores it serves
        - DC inventory = sum(served store inventory) × buffer_factor
        - Buffer factor accounts for safety stock and pipeline inventory
        """

        dc_inventory = []
        buffer_factor = 0.30  # DCs hold 30% of store inventory as buffer

        # Add dc_id to store inventory based on store_to_dc_map
        store_inv_with_dc = store_inventory_df.copy()
        store_inv_with_dc["dc_id"] = store_inv_with_dc["store_id"].map(
            store_to_dc_map
        )

        # Group by DC and product to aggregate inventory
        grouped = (
            store_inv_with_dc
            .groupby(["dc_id", "product_id"])
            ["inventory_qty"]
            .sum()
            .reset_index()
        )

        for _, row in grouped.iterrows():

            dc_inventory.append({
                "product_id":
                    row["product_id"],

                "dc_id":
                    row["dc_id"],

                "snapshot_date":
                    datetime.now().date(),

                "inventory_qty":
                    int(
                        row["inventory_qty"] * buffer_factor
                    ),

                "created_at":
                    datetime.now()
            })

        return pd.DataFrame(
            dc_inventory
        )